<a href="https://colab.research.google.com/github/Ssegawa-Carol/Wakiso-Smart-Garbage-Management-System/blob/master/Wakiso_Smart_Garbage_CW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Wakiso Smart Garbage Management System

import os
import sys
import json
import threading
import time
import random
import datetime
from datetime import datetime, timedelta
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Kill any existing processes on port 8000
print("🔄 Cleaning up existing processes...")
os.system('pkill -f "python.*manage.py" || true')
os.system('pkill -f "python.*runserver" || true')
os.system('pkill -f "python.*app.py" || true')
os.system('pkill -f ngrok || true')
os.system('fuser -k 8000/tcp || true')
time.sleep(2)

# Suppress all pip output
import contextlib
import io

def run_command(cmd, silent=True):
    if silent:
        with contextlib.redirect_stdout(io.StringIO()):
            with contextlib.redirect_stderr(io.StringIO()):
                return os.system(cmd)
    else:
        return os.system(cmd)

with contextlib.redirect_stdout(io.StringIO()):
    with contextlib.redirect_stderr(io.StringIO()):
        os.system('pip install flask -q')
        os.system('pip install flask-sqlalchemy -q')
        os.system('pip install flask-login -q')
        os.system('pip install flask-cors -q')
        os.system('pip install pandas -q')
        os.system('pip install numpy -q')
        os.system('pip install werkzeug -q')



os.chdir('/content')
os.makedirs('wakiso_smart_garbage_flask', exist_ok=True)
os.chdir('wakiso_smart_garbage_flask')

os.makedirs('templates', exist_ok=True)
os.makedirs('static', exist_ok=True)
os.makedirs('static/css', exist_ok=True)
os.makedirs('static/js', exist_ok=True)



# Create App.py

app_code = '''
import os
import json
import random
import math
from datetime import datetime, timedelta
from flask import Flask, render_template, request, redirect, url_for, flash, jsonify, session
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user
from flask_cors import CORS
from werkzeug.security import generate_password_hash, check_password_hash
from functools import wraps

# Create Flask app
app = Flask(__name__)
app.secret_key = 'your-secret-key-here-change-in-production'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///smart_garbage.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False
app.config['SESSION_COOKIE_SECURE'] = False
app.config['SESSION_COOKIE_HTTPONLY'] = True
app.config['SESSION_COOKIE_SAMESITE'] = 'Lax'

# Initialize extensions
db = SQLAlchemy(app)
login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_view = 'login'
login_manager.login_message = 'Please log in to access this page.'
CORS(app, supports_credentials=True, origins='*')


# ROLE-BASED ACCESS CONTROL DECORATOR

def role_required(allowed_roles):
    """Decorator to restrict access to specific roles"""
    def decorator(f):
        @wraps(f)
        def decorated_function(*args, **kwargs):
            if not current_user.is_authenticated:
                flash('Please log in to access this page.', 'warning')
                return redirect(url_for('login'))
            if current_user.role not in allowed_roles:
                flash('Access denied. You do not have permission for this action.', 'danger')
                return redirect(url_for('dashboard'))
            return f(*args, **kwargs)
        return decorated_function
    return decorator

# MODELS

class User(UserMixin, db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80), unique=True, nullable=False)
    password_hash = db.Column(db.String(200), nullable=False)
    role = db.Column(db.String(20), default='operator')
    email = db.Column(db.String(120), nullable=True)
    full_name = db.Column(db.String(100), nullable=True)
    is_active = db.Column(db.Boolean, default=True)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    updated_at = db.Column(db.DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    def set_password(self, password):
        self.password_hash = generate_password_hash(password)

    def check_password(self, password):
        return check_password_hash(self.password_hash, password)

    def get_role_display(self):
        role_display = {
            'admin': 'Administrator',
            'manager': 'Operations Manager',
            'operator': 'Control Room Operator',
            'driver': 'Driver'
        }
        return role_display.get(self.role, self.role)

    def get_role_color(self):
        colors = {
            'admin': 'danger',
            'manager': 'primary',
            'operator': 'info',
            'driver': 'success'
        }
        return colors.get(self.role, 'secondary')

class Region(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    name = db.Column(db.String(100), nullable=False)
    code = db.Column(db.String(10), unique=True, nullable=False)
    description = db.Column(db.Text)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

class Zone(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    name = db.Column(db.String(100), nullable=False)
    region_id = db.Column(db.Integer, db.ForeignKey('region.id'))
    code = db.Column(db.String(10), unique=True, nullable=False)
    description = db.Column(db.Text)
    center_lat = db.Column(db.Float, default=0.0)
    center_lng = db.Column(db.Float, default=0.0)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

    region = db.relationship('Region', backref='zones')

class Bin(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    bin_id = db.Column(db.String(50), unique=True, nullable=False)
    name = db.Column(db.String(100), nullable=False)
    bin_type = db.Column(db.String(20), default='STANDARD')
    capacity = db.Column(db.Float, default=1000)
    current_fill_level = db.Column(db.Float, default=0)
    fill_percentage = db.Column(db.Float, default=0)
    status = db.Column(db.String(20), default='ONLINE')
    latitude = db.Column(db.Float, nullable=False)
    longitude = db.Column(db.Float, nullable=False)
    address = db.Column(db.Text)
    zone_id = db.Column(db.Integer, db.ForeignKey('zone.id'))
    battery_level = db.Column(db.Float, default=100)
    temperature = db.Column(db.Float, nullable=True)
    humidity = db.Column(db.Float, nullable=True)
    last_maintenance = db.Column(db.DateTime, nullable=True)
    last_empty = db.Column(db.DateTime, nullable=True)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    updated_at = db.Column(db.DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    zone = db.relationship('Zone', backref='bins')

    def get_status_color(self):
        if self.fill_percentage <= 60:
            return 'green'
        elif self.fill_percentage <= 85:
            return 'amber'
        else:
            return 'red'

    def get_fill_level_class(self):
        if self.fill_percentage <= 60:
            return 'success'
        elif self.fill_percentage <= 85:
            return 'warning'
        else:
            return 'danger'

    def to_dict(self):
        return {
            'id': self.bin_id,
            'name': self.name,
            'fill': self.fill_percentage,
            'lat': self.latitude,
            'lng': self.longitude,
            'status': self.status,
            'status_color': self.get_status_color(),
            'address': self.address,
            'zone': self.zone.name if self.zone else 'Unknown',
            'battery': self.battery_level,
            'last_empty': self.last_empty.strftime('%Y-%m-%d %H:%M') if self.last_empty else None,
        }

class BinReading(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    bin_id = db.Column(db.Integer, db.ForeignKey('bin.id'))
    fill_level = db.Column(db.Float)
    fill_percentage = db.Column(db.Float)
    temperature = db.Column(db.Float, nullable=True)
    humidity = db.Column(db.Float, nullable=True)
    battery_level = db.Column(db.Float, nullable=True)
    recorded_at = db.Column(db.DateTime, default=datetime.utcnow)
    is_anomaly = db.Column(db.Boolean, default=False)

    bin = db.relationship('Bin', backref='readings')

class Truck(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    truck_id = db.Column(db.String(20), unique=True, nullable=False)
    name = db.Column(db.String(100), nullable=False)
    status = db.Column(db.String(20), default='IDLE')
    latitude = db.Column(db.Float, nullable=True)
    longitude = db.Column(db.Float, nullable=True)
    zone_id = db.Column(db.Integer, db.ForeignKey('zone.id'))
    driver_name = db.Column(db.String(100))
    speed = db.Column(db.Float, default=30)
    fuel_level = db.Column(db.Float, default=100)
    last_active = db.Column(db.DateTime, default=datetime.utcnow)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

    zone = db.relationship('Zone', backref='trucks')

    def get_status_color(self):
        colors = {'IDLE': 'blue', 'EN_ROUTE': 'green', 'COLLECTING': 'orange', 'OFF_DUTY': 'gray'}
        return colors.get(self.status, 'blue')

class Alert(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    bin_id = db.Column(db.Integer, db.ForeignKey('bin.id'), nullable=True)
    truck_id = db.Column(db.Integer, db.ForeignKey('truck.id'), nullable=True)
    alert_type = db.Column(db.String(20), nullable=False)
    priority = db.Column(db.String(10), default='MEDIUM')
    message = db.Column(db.Text, nullable=False)
    is_resolved = db.Column(db.Boolean, default=False)
    is_acknowledged = db.Column(db.Boolean, default=False)
    acknowledged_at = db.Column(db.DateTime, nullable=True)
    resolved_at = db.Column(db.DateTime, nullable=True)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

    bin = db.relationship('Bin', backref='alerts')
    truck = db.relationship('Truck', backref='alerts')

# USER LOADER

@login_manager.user_loader
def load_user(user_id):
    return User.query.get(int(user_id))

# AUTHENTICATION ROUTES

@app.route('/')
def index():
    if current_user.is_authenticated:
        return redirect(url_for('dashboard'))
    return redirect(url_for('login'))

@app.route('/login', methods=['GET', 'POST'])
def login():
    if current_user.is_authenticated:
        return redirect(url_for('dashboard'))

    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        user = User.query.filter_by(username=username).first()

        if user and user.check_password(password):
            if not user.is_active:
                flash('Your account has been deactivated. Please contact an administrator.', 'danger')
                return render_template('login.html')
            login_user(user)
            next_page = request.args.get('next')
            flash(f'Welcome back, {user.full_name or user.username}!', 'success')
            return redirect(next_page or url_for('dashboard'))
        else:
            flash('Invalid username or password.', 'danger')

    return render_template('login.html')

@app.route('/logout')
@login_required
def logout():
    logout_user()
    flash('You have been logged out.', 'info')
    return redirect(url_for('login'))


# ADMIN PANEL ROUTES

@app.route('/admin')
@login_required
@role_required(['admin'])
def admin_panel():
    users = User.query.all()
    total_users = len(users)
    active_users = len([u for u in users if u.is_active])
    roles_count = {
        'admin': len([u for u in users if u.role == 'admin']),
        'manager': len([u for u in users if u.role == 'manager']),
        'operator': len([u for u in users if u.role == 'operator']),
        'driver': len([u for u in users if u.role == 'driver']),
    }
    return render_template('admin.html',
                         users=users,
                         total_users=total_users,
                         active_users=active_users,
                         roles_count=roles_count)

@app.route('/admin/users')
@login_required
@role_required(['admin'])
def admin_users():
    users = User.query.all()
    return render_template('admin_users.html', users=users)

@app.route('/admin/user/add', methods=['GET', 'POST'])
@login_required
@role_required(['admin'])
def admin_add_user():
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        role = request.form.get('role')
        full_name = request.form.get('full_name')
        email = request.form.get('email')

        # Validate inputs
        if not username or not password:
            flash('Username and password are required.', 'danger')
            return render_template('admin_add_user.html')

        # Check if user exists
        if User.query.filter_by(username=username).first():
            flash('Username already exists.', 'danger')
            return render_template('admin_add_user.html')

        # Create user
        user = User(
            username=username,
            role=role,
            full_name=full_name,
            email=email,
            is_active=True
        )
        user.set_password(password)
        db.session.add(user)
        db.session.commit()

        flash(f'User {username} created successfully!', 'success')
        return redirect(url_for('admin_users'))

    return render_template('admin_add_user.html')

@app.route('/admin/user/edit/<int:user_id>', methods=['GET', 'POST'])
@login_required
@role_required(['admin'])
def admin_edit_user(user_id):
    user = User.query.get_or_404(user_id)

    # Prevent editing yourself out of admin
    if user.id == current_user.id:
        flash('You cannot edit your own account here. Use profile settings.', 'warning')
        return redirect(url_for('admin_users'))

    if request.method == 'POST':
        user.username = request.form.get('username')
        user.role = request.form.get('role')
        user.full_name = request.form.get('full_name')
        user.email = request.form.get('email')
        user.is_active = request.form.get('is_active') == 'on'

        # Update password if provided
        new_password = request.form.get('password')
        if new_password:
            user.set_password(new_password)

        db.session.commit()
        flash(f'User {user.username} updated successfully!', 'success')
        return redirect(url_for('admin_users'))

    return render_template('admin_edit_user.html', user=user)

@app.route('/admin/user/delete/<int:user_id>', methods=['POST'])
@login_required
@role_required(['admin'])
def admin_delete_user(user_id):
    user = User.query.get_or_404(user_id)

    # Prevent deleting yourself
    if user.id == current_user.id:
        flash('You cannot delete your own account.', 'danger')
        return redirect(url_for('admin_users'))

    username = user.username
    db.session.delete(user)
    db.session.commit()
    flash(f'User {username} deleted successfully.', 'success')
    return redirect(url_for('admin_users'))

@app.route('/admin/user/toggle/<int:user_id>', methods=['POST'])
@login_required
@role_required(['admin'])
def admin_toggle_user(user_id):
    user = User.query.get_or_404(user_id)

    # Prevent toggling yourself
    if user.id == current_user.id:
        flash('You cannot deactivate your own account.', 'danger')
        return redirect(url_for('admin_users'))

    user.is_active = not user.is_active
    db.session.commit()
    status = 'activated' if user.is_active else 'deactivated'
    flash(f'User {user.username} {status}.', 'success')
    return redirect(url_for('admin_users'))

# ============================================================================
# MAIN APPLICATION ROUTES
# ============================================================================

@app.route('/dashboard')
@login_required
def dashboard():
    total_bins = Bin.query.count()
    critical_bins = Bin.query.filter(Bin.fill_percentage > 85).count()
    trucks_on_road = Truck.query.filter(Truck.status != 'OFF_DUTY').count()
    total_trucks = Truck.query.count()

    avg_fill = db.session.query(db.func.avg(Bin.fill_percentage)).scalar() or 0
    active_alerts = Alert.query.filter_by(is_resolved=False).count()
    critical_alerts = Alert.query.filter_by(is_resolved=False, priority='CRITICAL').count()

    context = {
        'total_bins': total_bins,
        'critical_bins': critical_bins,
        'trucks_on_road': trucks_on_road,
        'total_trucks': total_trucks,
        'avg_fill': round(avg_fill, 1),
        'active_alerts': active_alerts,
        'critical_alerts': critical_alerts,
        'uptime': 99.8,
    }
    return render_template('dashboard.html', **context)

@app.route('/bins')
@login_required
def bin_list():
    bins = Bin.query.all()
    zones = Zone.query.all()
    return render_template('bin_list.html', bins=bins, zones=zones)

@app.route('/bin/<bin_id>')
@login_required
def bin_detail(bin_id):
    bin_obj = Bin.query.filter_by(bin_id=bin_id).first_or_404()
    readings = BinReading.query.filter_by(bin_id=bin_obj.id).order_by(BinReading.recorded_at.desc()).limit(24).all()
    alerts = Alert.query.filter_by(bin_id=bin_obj.id, is_resolved=False).all()
    return render_template('bin_detail.html', bin=bin_obj, readings=readings, alerts=alerts)

@app.route('/map')
@login_required
def map_view():
    bins = Bin.query.all()
    trucks = Truck.query.all()

    bins_json = json.dumps([b.to_dict() for b in bins])
    trucks_json = json.dumps([{
        'id': t.truck_id,
        'name': t.name,
        'lat': t.latitude if t.latitude else None,
        'lng': t.longitude if t.longitude else None,
        'status': t.status,
        'status_color': t.get_status_color(),
        'driver': t.driver_name,
        'speed': t.speed,
    } for t in trucks])

    return render_template('map.html',
                         bins_json=bins_json,
                         trucks_json=trucks_json,
                         total_bins=len(bins),
                         total_trucks=len(trucks))

@app.route('/actions')
@login_required
def actions_view():
    return render_template('actions.html')

# ============================================================================
# API ENDPOINTS
# ============================================================================

@app.route('/api/dashboard-data')
@login_required
def dashboard_data():
    status_counts = {
        'ONLINE': Bin.query.filter_by(status='ONLINE').count(),
        'OFFLINE': Bin.query.filter_by(status='OFFLINE').count(),
        'ERROR': Bin.query.filter_by(status='ERROR').count(),
    }

    alert_types = {}
    for at in ['FULL', 'MAINTENANCE', 'ANOMALY', 'BATTERY_LOW', 'OVERFLOW', 'TRUCK_DELAY']:
        alert_types[at] = Alert.query.filter_by(alert_type=at, is_resolved=False).count()

    recent_readings = []
    for bin in Bin.query.limit(5).all():
        for reading in BinReading.query.filter_by(bin_id=bin.id).order_by(BinReading.recorded_at.desc()).limit(10).all():
            recent_readings.append({
                'bin': bin.name,
                'fill_percentage': reading.fill_percentage,
                'recorded_at': reading.recorded_at.strftime('%Y-%m-%d %H:%M')
            })

    truck_status = {
        'IDLE': Truck.query.filter_by(status='IDLE').count(),
        'EN_ROUTE': Truck.query.filter_by(status='EN_ROUTE').count(),
        'COLLECTING': Truck.query.filter_by(status='COLLECTING').count(),
        'OFF_DUTY': Truck.query.filter_by(status='OFF_DUTY').count(),
    }

    return jsonify({
        'status_counts': status_counts,
        'alert_types': alert_types,
        'recent_readings': recent_readings,
        'truck_status': truck_status,
    })

@app.route('/api/map-data')
@login_required
def map_data():
    bins_data = []
    for bin in Bin.query.all():
        bins_data.append({
            'id': bin.bin_id,
            'name': bin.name,
            'lat': bin.latitude,
            'lng': bin.longitude,
            'fill': bin.fill_percentage,
            'status': bin.status,
            'status_color': bin.get_status_color(),
            'address': bin.address,
            'zone': bin.zone.name if bin.zone else 'Unknown',
            'battery': bin.battery_level,
            'last_empty': bin.last_empty.strftime('%Y-%m-%d %H:%M') if bin.last_empty else None,
        })

    trucks_data = []
    for truck in Truck.query.all():
        if truck.latitude and truck.longitude:
            trucks_data.append({
                'id': truck.truck_id,
                'name': truck.name,
                'lat': truck.latitude,
                'lng': truck.longitude,
                'status': truck.status,
                'status_color': truck.get_status_color(),
                'driver': truck.driver_name,
                'speed': truck.speed,
            })

    return jsonify({
        'bins': bins_data,
        'trucks': trucks_data,
        'total_bins': len(bins_data),
        'total_trucks': len(trucks_data),
    })

@app.route('/api/actions-data')
@login_required
def actions_data():
    critical_bins = Bin.query.filter(Bin.fill_percentage > 85).order_by(Bin.fill_percentage.desc()).limit(20).all()
    trucks = Truck.query.filter_by(status='IDLE').all()

    actions = []
    for bin in critical_bins:
        nearest_truck = None
        min_distance = float('inf')
        for truck in trucks:
            if truck.latitude and truck.longitude:
                dist = math.sqrt((bin.latitude - truck.latitude)**2 + (bin.longitude - truck.longitude)**2)
                if dist < min_distance:
                    min_distance = dist
                    nearest_truck = truck.truck_id

        priority = 'CRITICAL' if bin.fill_percentage > 95 else 'HIGH' if bin.fill_percentage > 85 else 'MEDIUM'
        actions.append({
            'bin_id': bin.bin_id,
            'name': bin.name,
            'fill': round(bin.fill_percentage, 1),
            'address': bin.address,
            'priority': priority,
            'assigned_truck': nearest_truck or 'None',
            'zone': bin.zone.name if bin.zone else 'Unknown',
        })

    return jsonify({'actions': actions[:10]})

@app.route('/api/alerts-feed')
@login_required
def alerts_feed():
    alerts = Alert.query.filter_by(is_resolved=False).order_by(Alert.created_at.desc()).limit(20).all()
    data = []
    for alert in alerts:
        source = 'System'
        if alert.bin:
            source = alert.bin.name
        elif alert.truck:
            source = alert.truck.truck_id

        data.append({
            'id': alert.id,
            'type': alert.alert_type,
            'priority': alert.priority,
            'message': alert.message,
            'created_at': alert.created_at.strftime('%Y-%m-%d %H:%M:%S'),
            'is_acknowledged': alert.is_acknowledged,
            'source': source,
        })
    return jsonify({'alerts': data})

@app.route('/api/acknowledge/<int:alert_id>', methods=['POST'])
@login_required
def acknowledge_alert(alert_id):
    alert = Alert.query.get_or_404(alert_id)
    alert.is_acknowledged = True
    alert.acknowledged_at = datetime.utcnow()
    db.session.commit()
    return jsonify({'status': 'acknowledged'})

@app.route('/api/simulate', methods=['POST'])
@login_required
def simulate_data():
    # Simulate bin readings
    for bin in Bin.query.all():
        change = random.uniform(-8, 12)
        new_fill = max(0, min(bin.capacity, bin.current_fill_level + change))
        battery_change = random.uniform(-3, 1)
        new_battery = max(0, min(100, bin.battery_level + battery_change))

        reading = BinReading(
            bin_id=bin.id,
            fill_level=new_fill,
            fill_percentage=(new_fill / bin.capacity) * 100,
            temperature=random.uniform(20, 38),
            humidity=random.uniform(40, 85),
            battery_level=new_battery,
        )
        db.session.add(reading)

        bin.current_fill_level = new_fill
        bin.battery_level = new_battery

        if reading.fill_percentage > 95:
            alert = Alert(
                bin_id=bin.id,
                alert_type='OVERFLOW',
                priority='CRITICAL',
                message=f'Bin {bin.name} is OVERFLOWING at {reading.fill_percentage:.1f}% - IMMEDIATE ACTION REQUIRED!'
            )
            db.session.add(alert)
        elif reading.fill_percentage > 85:
            alert = Alert(
                bin_id=bin.id,
                alert_type='FULL',
                priority='HIGH',
                message=f'Bin {bin.name} is at {reading.fill_percentage:.1f}% capacity - Schedule pickup urgently'
            )
            db.session.add(alert)

        if new_battery < 15:
            alert = Alert(
                bin_id=bin.id,
                alert_type='BATTERY_LOW',
                priority='MEDIUM',
                message=f'Bin {bin.name} battery at {new_battery:.1f}% - Replace battery soon'
            )
            db.session.add(alert)

    # Simulate truck movements
    for truck in Truck.query.all():
        if truck.status in ['EN_ROUTE', 'COLLECTING']:
            if truck.latitude and truck.longitude:
                truck.latitude += random.uniform(-0.005, 0.005)
                truck.longitude += random.uniform(-0.005, 0.005)

        if random.random() < 0.1:
            truck.status = random.choice(['IDLE', 'EN_ROUTE', 'COLLECTING'])

    db.session.commit()
    return jsonify({'status': 'Data simulated successfully'})

# CREATE DATABASE AND SEED DATA

def create_database():
    with app.app_context():
        db.create_all()

        # Check if data exists
        if User.query.count() > 0:
            print("Data already exists, skipping seed.")
            return



        # Create admin user
        admin = User(
            username='admin',
            role='admin',
            full_name='System Administrator',
            email='admin@wakiso-garbage.com',
            is_active=True
        )
        admin.set_password('Nebraska@2011')
        db.session.add(admin)

        # Create sample users for demonstration
        sample_users = [
            {'username': 'manager', 'password': 'manager123', 'role': 'manager', 'full_name': 'Operations Manager', 'email': 'manager@wakiso-garbage.com'},
            {'username': 'operator', 'password': 'operator123', 'role': 'operator', 'full_name': 'Control Room Operator', 'email': 'operator@wakiso-garbage.com'},
            {'username': 'driver', 'password': 'driver123', 'role': 'driver', 'full_name': 'Fleet Driver', 'email': 'driver@wakiso-garbage.com'},
        ]

        for user_data in sample_users:
            user = User(
                username=user_data['username'],
                role=user_data['role'],
                full_name=user_data['full_name'],
                email=user_data['email'],
                is_active=True
            )
            user.set_password(user_data['password'])
            db.session.add(user)

        db.session.commit()

        # Create regions
        regions = [
            Region(name='Central Wakiso', code='CW', description='Central region'),
            Region(name='North Wakiso', code='NW', description='Northern region'),
        ]
        for r in regions:
            db.session.add(r)
        db.session.commit()

        # Create zones with proper coordinates
        areas = [
            {'name': 'Nansana', 'region_id': 1, 'center_lat': 0.3640, 'center_lng': 32.5280},
            {'name': 'Kira', 'region_id': 1, 'center_lat': 0.3976, 'center_lng': 32.6225},
            {'name': 'Makindye', 'region_id': 1, 'center_lat': 0.3032, 'center_lng': 32.5930},
            {'name': 'Katabi', 'region_id': 2, 'center_lat': 0.1860, 'center_lng': 32.5150},
            {'name': 'Kiteezi', 'region_id': 2, 'center_lat': 0.4430, 'center_lng': 32.5660},
        ]

        zones = []
        for area in areas:
            zone = Zone(
                name=area['name'],
                region_id=area['region_id'],
                code=area['name'][:4].upper(),
                description=f"{area['name']} zone",
                center_lat=area['center_lat'],
                center_lng=area['center_lng']
            )
            db.session.add(zone)
            zones.append(zone)
        db.session.commit()

        # Create bins with proper coordinates
        bin_types = ['STANDARD', 'RECYCLING', 'ORGANIC', 'MEDICAL', 'ELECTRONIC']
        locations = ['Main', 'Central', 'Junction', 'Market', 'Square', 'Plaza', 'Complex', 'Station', 'Centre', 'Point', 'East', 'West']

        for zone_idx, zone in enumerate(zones):
            area = areas[zone_idx]
            for i in range(12):
                fill = random.uniform(5, 98)
                status = 'ONLINE' if random.random() > 0.12 else ('OFFLINE' if random.random() > 0.5 else 'ERROR')
                lat_offset = random.uniform(-0.015, 0.015)
                lng_offset = random.uniform(-0.015, 0.015)
                location = random.choice(locations)

                bin_obj = Bin(
                    bin_id=f"BIN{len(Bin.query.all()) + 1:03d}",
                    name=f"{area['name']} {location} Bin",
                    bin_type=random.choice(bin_types),
                    capacity=random.choice([800, 1000, 1200, 1500, 2000]),
                    current_fill_level=fill * 10,
                    fill_percentage=round(fill, 2),
                    status=status,
                    latitude=round(area['center_lat'] + lat_offset, 7),
                    longitude=round(area['center_lng'] + lng_offset, 7),
                    address=f"{location}, {area['name']}, Wakiso",
                    zone_id=zone.id,
                    battery_level=round(random.uniform(20, 100), 1),
                    temperature=round(random.uniform(20, 38), 1),
                    humidity=round(random.uniform(40, 85), 1),
                    last_empty=datetime.utcnow() - timedelta(days=random.randint(0, 5)) if random.random() > 0.3 else None,
                    last_maintenance=datetime.utcnow() - timedelta(days=random.randint(0, 30)) if random.random() > 0.4 else None,
                )
                db.session.add(bin_obj)
        db.session.commit()

        # Create readings for each bin
        for bin in Bin.query.all():
            for h in range(24):
                base_fill = random.uniform(10, 90)
                reading = BinReading(
                    bin_id=bin.id,
                    fill_level=base_fill * 10,
                    fill_percentage=round(base_fill, 2),
                    temperature=round(random.uniform(20, 35), 1),
                    humidity=round(random.uniform(40, 80), 1),
                    battery_level=round(random.uniform(60, 100), 1),
                    recorded_at=datetime.utcnow() - timedelta(hours=h),
                )
                db.session.add(reading)
        db.session.commit()

        # Create trucks with proper coordinates
        truck_statuses = ['IDLE', 'EN_ROUTE', 'COLLECTING', 'OFF_DUTY']
        drivers = ['John M.', 'Sarah K.', 'David L.', 'Grace M.', 'Robert O.', 'Mary N.', 'Peter S.', 'Jane K.', 'Joseph E.', 'Alice T.']

        truck_pk = 1
        for zone in zones:
            for j in range(3):
                status = random.choice(truck_statuses)
                lat_offset = random.uniform(-0.01, 0.01)
                lng_offset = random.uniform(-0.01, 0.01)
                lat = zone.center_lat + lat_offset if status != 'OFF_DUTY' else None
                lng = zone.center_lng + lng_offset if status != 'OFF_DUTY' else None

                truck = Truck(
                    truck_id=f"WK-{truck_pk:03d}",
                    name=f"Truck {truck_pk}",
                    status=status,
                    latitude=round(lat, 7) if lat else None,
                    longitude=round(lng, 7) if lng else None,
                    zone_id=zone.id,
                    driver_name=random.choice(drivers),
                    speed=random.choice([0, 15, 25, 30, 40]),
                    fuel_level=round(random.uniform(30, 100), 1),
                )
                db.session.add(truck)
                truck_pk += 1
        db.session.commit()

        # Create alerts
        alert_types = ['FULL', 'MAINTENANCE', 'ANOMALY', 'BATTERY_LOW', 'OVERFLOW']
        priorities = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

        bins_list = Bin.query.all()
        for i in range(40):
            bin_obj = random.choice(bins_list)
            alert = Alert(
                bin_id=bin_obj.id,
                alert_type=random.choice(alert_types),
                priority=random.choice(priorities),
                message=f"Alert for bin {bin_obj.name}",
                is_resolved=random.random() > 0.5,
                is_acknowledged=random.random() > 0.6,
                created_at=datetime.utcnow() - timedelta(minutes=random.randint(0, 120)),
            )
            db.session.add(alert)
        db.session.commit()

        print(f"Data loaded successfully!")
        print(f"Users: {User.query.count()}")
        print(f"Regions: {Region.query.count()}")
        print(f"Zones: {Zone.query.count()}")
        print(f"Bins: {Bin.query.count()}")
        print(f"Trucks: {Truck.query.count()}")
        print(f"Alerts: {Alert.query.count()}")
        print(f"BinReadings: {BinReading.query.count()}")

# RUN THE APP

if __name__ == '__main__':
    create_database()
    print("🚀 Starting Flask server on port 8000...")
    app.run(host='0.0.0.0', port=8000, debug=False, threaded=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)



# CREATE ADMIN TEMPLATES

# Admin Dashboard Template
admin_html = '''
{% extends 'base.html' %}
{% block title %}Admin Panel{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-user-shield me-2"></i>Admin Panel</h1>
<a href="{{ url_for('admin_users') }}" class="btn btn-primary">
    <i class="fas fa-users me-2"></i>Manage Users
</a>
</div>

<div class="row">
<div class="col-md-3 mb-3">
<div class="card bg-primary text-white">
<div class="card-body">
<h5 class="card-title">Total Users</h5>
<h2 class="mb-0">{{ total_users }}</h2>
</div>
</div>
</div>
<div class="col-md-3 mb-3">
<div class="card bg-success text-white">
<div class="card-body">
<h5 class="card-title">Active Users</h5>
<h2 class="mb-0">{{ active_users }}</h2>
</div>
</div>
</div>
<div class="col-md-3 mb-3">
<div class="card bg-danger text-white">
<div class="card-body">
<h5 class="card-title">Administrators</h5>
<h2 class="mb-0">{{ roles_count.admin }}</h2>
</div>
</div>
</div>
<div class="col-md-3 mb-3">
<div class="card bg-info text-white">
<div class="card-body">
<h5 class="card-title">Other Roles</h5>
<h2 class="mb-0">{{ roles_count.manager + roles_count.operator + roles_count.driver }}</h2>
</div>
</div>
</div>
</div>

<div class="row">
<div class="col-md-6">
<div class="card">
<div class="card-header">
<h6><i class="fas fa-chart-pie me-2"></i>User Role Distribution</h6>
</div>
<div class="card-body">
<canvas id="roleChart" height="200"></canvas>
</div>
</div>
</div>
<div class="col-md-6">
<div class="card">
<div class="card-header">
<h6><i class="fas fa-users me-2"></i>Quick Actions</h6>
</div>
<div class="card-body">
<div class="d-grid gap-2">
<a href="{{ url_for('admin_add_user') }}" class="btn btn-success">
    <i class="fas fa-user-plus me-2"></i>Add New User
</a>
<a href="{{ url_for('admin_users') }}" class="btn btn-primary">
    <i class="fas fa-users me-2"></i>View All Users
</a>
<a href="#" class="btn btn-info" onclick="location.reload()">
    <i class="fas fa-sync me-2"></i>Refresh Data
</a>
</div>
</div>
</div>
</div>
</div>

<div class="row mt-3">
<div class="col-12">
<div class="card">
<div class="card-header">
<h6><i class="fas fa-info-circle me-2"></i>System Information</h6>
</div>
<div class="card-body">
<div class="row">
<div class="col-md-6">
<p><strong>System Version:</strong> 1.0.0</p>
<p><strong>Database:</strong> SQLite</p>
<p><strong>Users:</strong> {{ total_users }}</p>
</div>
<div class="col-md-6">
<p><strong>Roles Supported:</strong> Admin, Manager, Operator, Driver</p>
<p><strong>Current User:</strong> {{ current_user.username }} ({{ current_user.get_role_display() }})</p>
<p><strong>Session Expires:</strong> 8 hours</p>
</div>
</div>
</div>
</div>
</div>
</div>
</div>

<script>
document.addEventListener('DOMContentLoaded', function() {
    const ctx = document.getElementById('roleChart').getContext('2d');
    new Chart(ctx, {
        type: 'doughnut',
        data: {
            labels: ['Administrator', 'Manager', 'Operator', 'Driver'],
            datasets: [{
                data: [
                    {{ roles_count.admin }},
                    {{ roles_count.manager }},
                    {{ roles_count.operator }},
                    {{ roles_count.driver }}
                ],
                backgroundColor: ['#dc3545', '#0d6efd', '#0dcaf0', '#198754']
            }]
        },
        options: {
            responsive: true,
            plugins: {
                legend: {
                    position: 'bottom'
                }
            }
        }
    });
});
</script>
{% endblock %}
'''

with open('templates/admin.html', 'w') as f:
    f.write(admin_html)


# Admin Users List Template
admin_users_html = '''
{% extends 'base.html' %}
{% block title %}Manage Users{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-users me-2"></i>Manage Users</h1>
<a href="{{ url_for('admin_add_user') }}" class="btn btn-success">
    <i class="fas fa-user-plus me-2"></i>Add User
</a>
</div>

{% with messages = get_flashed_messages(with_categories=true) %}
    {% if messages %}
        {% for category, message in messages %}
            <div class="alert alert-{{ category }} alert-dismissible fade show">
                {{ message }}
                <button type="button" class="btn-close" data-bs-dismiss="alert"></button>
            </div>
        {% endfor %}
    {% endif %}
{% endwith %}

<div class="card">
<div class="card-body">
<div class="table-responsive">
<table class="table table-hover">
    <thead>
        <tr>
            <th>Username</th>
            <th>Full Name</th>
            <th>Email</th>
            <th>Role</th>
            <th>Status</th>
            <th>Created</th>
            <th>Actions</th>
        </tr>
    </thead>
    <tbody>
        {% for user in users %}
        <tr>
            <td><strong>{{ user.username }}</strong></td>
            <td>{{ user.full_name or '-' }}</td>
            <td>{{ user.email or '-' }}</td>
            <td>
                <span class="badge bg-{{ user.get_role_color() }}">
                    {{ user.get_role_display() }}
                </span>
            </td>
            <td>
                {% if user.is_active %}
                    <span class="badge bg-success">Active</span>
                {% else %}
                    <span class="badge bg-secondary">Inactive</span>
                {% endif %}
            </td>
            <td>{{ user.created_at.strftime('%Y-%m-%d') }}</td>
            <td>
                <div class="btn-group btn-group-sm">
                    <a href="{{ url_for('admin_edit_user', user_id=user.id) }}" class="btn btn-primary">
                        <i class="fas fa-edit"></i>
                    </a>
                    {% if user.id != current_user.id %}
                        <form method="POST" action="{{ url_for('admin_toggle_user', user_id=user.id) }}" style="display:inline;">
                            <button type="submit" class="btn btn-{{ 'warning' if user.is_active else 'success' }}"
                                    onclick="return confirm('{{ 'Deactivate' if user.is_active else 'Activate' }} user {{ user.username }}?')">
                                <i class="fas fa-{{ 'times' if user.is_active else 'check' }}"></i>
                            </button>
                        </form>
                        <form method="POST" action="{{ url_for('admin_delete_user', user_id=user.id) }}" style="display:inline;">
                            <button type="submit" class="btn btn-danger"
                                    onclick="return confirm('Delete user {{ user.username }}? This action cannot be undone.')">
                                <i class="fas fa-trash"></i>
                            </button>
                        </form>
                    {% endif %}
                </div>
            </td>
        </tr>
        {% endfor %}
    </tbody>
</table>
</div>
</div>
</div>
</div>
{% endblock %}
'''

with open('templates/admin_users.html', 'w') as f:
    f.write(admin_users_html)


# Admin Add User Template
admin_add_user_html = '''
{% extends 'base.html' %}
{% block title %}Add User{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-user-plus me-2"></i>Add New User</h1>
<a href="{{ url_for('admin_users') }}" class="btn btn-secondary">
    <i class="fas fa-arrow-left me-2"></i>Back to Users
</a>
</div>

{% with messages = get_flashed_messages(with_categories=true) %}
    {% if messages %}
        {% for category, message in messages %}
            <div class="alert alert-{{ category }} alert-dismissible fade show">
                {{ message }}
                <button type="button" class="btn-close" data-bs-dismiss="alert"></button>
            </div>
        {% endfor %}
    {% endif %}
{% endwith %}

<div class="row">
<div class="col-md-6">
<div class="card">
<div class="card-body">
<form method="POST">
    <div class="mb-3">
        <label for="username" class="form-label">Username <span class="text-danger">*</span></label>
        <input type="text" class="form-control" id="username" name="username" required>
        <small class="text-muted">Unique username for login</small>
    </div>
    <div class="mb-3">
        <label for="password" class="form-label">Password <span class="text-danger">*</span></label>
        <input type="password" class="form-control" id="password" name="password" required>
        <small class="text-muted">Minimum 8 characters with uppercase, lowercase, and numbers</small>
    </div>
    <div class="mb-3">
        <label for="full_name" class="form-label">Full Name</label>
        <input type="text" class="form-control" id="full_name" name="full_name">
        <small class="text-muted">User's full name (optional)</small>
    </div>
    <div class="mb-3">
        <label for="email" class="form-label">Email</label>
        <input type="email" class="form-control" id="email" name="email">
        <small class="text-muted">Email address (optional)</small>
    </div>
    <div class="mb-3">
        <label for="role" class="form-label">Role <span class="text-danger">*</span></label>
        <select class="form-select" id="role" name="role" required>
            <option value="admin">Administrator - Full system access</option>
            <option value="manager" selected>Operations Manager - Oversee all operations</option>
            <option value="operator">Control Room Operator - Monitor and dispatch</option>
            <option value="driver">Driver - View assigned routes</option>
        </select>
    </div>
    <div class="d-grid">
        <button type="submit" class="btn btn-primary">
            <i class="fas fa-save me-2"></i>Create User
        </button>
    </div>
</form>
</div>
</div>
</div>
<div class="col-md-6">
<div class="card">
<div class="card-header">
<h6><i class="fas fa-info-circle me-2"></i>Role Descriptions</h6>
</div>
<div class="card-body">
<div class="list-group">
    <div class="list-group-item">
        <h6 class="mb-1"><span class="badge bg-danger">Admin</span> Administrator</h6>
        <p class="mb-1 text-muted">Full system access, user management, system configuration</p>
    </div>
    <div class="list-group-item">
        <h6 class="mb-1"><span class="badge bg-primary">Manager</span> Operations Manager</h6>
        <p class="mb-1 text-muted">Oversee all operations, dispatch decisions, route optimization</p>
    </div>
    <div class="list-group-item">
        <h6 class="mb-1"><span class="badge bg-info">Operator</span> Control Room Operator</h6>
        <p class="mb-1 text-muted">Real-time monitoring, dispatch trucks, acknowledge alerts</p>
    </div>
    <div class="list-group-item">
        <h6 class="mb-1"><span class="badge bg-success">Driver</span> Driver</h6>
        <p class="mb-1 text-muted">View assigned routes, update collection status (mobile)</p>
    </div>
</div>
</div>
</div>
</div>
</div>
</div>
{% endblock %}
'''

with open('templates/admin_add_user.html', 'w') as f:
    f.write(admin_add_user_html)


# Admin Edit User Template
admin_edit_user_html = '''
{% extends 'base.html' %}
{% block title %}Edit User{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-user-edit me-2"></i>Edit User: {{ user.username }}</h1>
<a href="{{ url_for('admin_users') }}" class="btn btn-secondary">
    <i class="fas fa-arrow-left me-2"></i>Back to Users
</a>
</div>

{% with messages = get_flashed_messages(with_categories=true) %}
    {% if messages %}
        {% for category, message in messages %}
            <div class="alert alert-{{ category }} alert-dismissible fade show">
                {{ message }}
                <button type="button" class="btn-close" data-bs-dismiss="alert"></button>
            </div>
        {% endfor %}
    {% endif %}
{% endwith %}

<div class="row">
<div class="col-md-6">
<div class="card">
<div class="card-body">
<form method="POST">
    <div class="mb-3">
        <label for="username" class="form-label">Username <span class="text-danger">*</span></label>
        <input type="text" class="form-control" id="username" name="username" value="{{ user.username }}" required>
    </div>
    <div class="mb-3">
        <label for="password" class="form-label">New Password</label>
        <input type="password" class="form-control" id="password" name="password">
        <small class="text-muted">Leave blank to keep current password</small>
    </div>
    <div class="mb-3">
        <label for="full_name" class="form-label">Full Name</label>
        <input type="text" class="form-control" id="full_name" name="full_name" value="{{ user.full_name or '' }}">
    </div>
    <div class="mb-3">
        <label for="email" class="form-label">Email</label>
        <input type="email" class="form-control" id="email" name="email" value="{{ user.email or '' }}">
    </div>
    <div class="mb-3">
        <label for="role" class="form-label">Role <span class="text-danger">*</span></label>
        <select class="form-select" id="role" name="role" required>
            <option value="admin" {% if user.role == 'admin' %}selected{% endif %}>Administrator</option>
            <option value="manager" {% if user.role == 'manager' %}selected{% endif %}>Operations Manager</option>
            <option value="operator" {% if user.role == 'operator' %}selected{% endif %}>Control Room Operator</option>
            <option value="driver" {% if user.role == 'driver' %}selected{% endif %}>Driver</option>
        </select>
    </div>
    <div class="mb-3 form-check">
        <input type="checkbox" class="form-check-input" id="is_active" name="is_active" {% if user.is_active %}checked{% endif %}>
        <label class="form-check-label" for="is_active">Active</label>
        <small class="text-muted d-block">Inactive users cannot log in</small>
    </div>
    <div class="d-grid">
        <button type="submit" class="btn btn-primary">
            <i class="fas fa-save me-2"></i>Update User
        </button>
    </div>
</form>
</div>
</div>
</div>
<div class="col-md-6">
<div class="card">
<div class="card-header">
<h6><i class="fas fa-info-circle me-2"></i>User Information</h6>
</div>
<div class="card-body">
<p><strong>Created:</strong> {{ user.created_at.strftime('%Y-%m-%d %H:%M') }}</p>
<p><strong>Last Updated:</strong> {{ user.updated_at.strftime('%Y-%m-%d %H:%M') }}</p>
<p><strong>Role:</strong> <span class="badge bg-{{ user.get_role_color() }}">{{ user.get_role_display() }}</span></p>
<p><strong>Status:</strong> {% if user.is_active %}<span class="badge bg-success">Active</span>{% else %}<span class="badge bg-secondary">Inactive</span>{% endif %}</p>
</div>
</div>
</div>
</div>
</div>
{% endblock %}
'''

with open('templates/admin_edit_user.html', 'w') as f:
    f.write(admin_edit_user_html)


# CREATE BASE.HTML WITH ADMIN LINK

base_html = '''
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Wakiso Smart Garbage{% endblock %}</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        :root{--primary:#2c3e50;--secondary:#34495e;--success:#2ecc71;--warning:#f39c12;--danger:#e74c3c;--info:#3498db;}
        .sidebar{min-height:100vh;background:linear-gradient(180deg,var(--primary) 0%,var(--secondary) 100%);}
        .sidebar .nav-link{color:#ecf0f1;}
        .sidebar .nav-link:hover{background:#34495e;color:#fff;}
        .sidebar .nav-link.active{background:var(--info);color:#fff;}
        .main-content{background:#f5f7fa;min-height:100vh;padding:20px;}
        .stats-card{border-radius:10px;padding:20px;box-shadow:0 2px 10px rgba(0,0,0,0.1);transition:transform 0.3s;cursor:pointer;background:white;}
        .stats-card:hover{transform:translateY(-5px);}
        .stats-icon{font-size:2.5rem;opacity:0.5;}
        .user-info{color:#ecf0f1;padding:10px 15px;border-bottom:1px solid #3d566e;}
        .user-info .username{font-weight:600;}
        .user-info .logout-link{color:#e74c3c;text-decoration:none;font-size:14px;}
        .user-info .logout-link:hover{color:#c0392b;text-decoration:underline;}
    </style>
    {% block extra_css %}{% endblock %}
</head>
<body>
<div class="container-fluid"><div class="row">
<nav class="col-md-3 col-lg-2 d-md-block sidebar p-0">
<div class="position-sticky">
    <div class="p-3 text-white">
        <h3><i class="fas fa-trash-alt me-2"></i> Wakiso Smart Garbage</h3>

    </div>
    <div class="user-info">
        <div><i class="fas fa-user-circle me-2"></i><span class="username">{{ current_user.username }}</span></div>
        <div><small class="text-muted">Role: {{ current_user.get_role_display() }}</small></div>
        <a href="{{ url_for('logout') }}" class="logout-link"><i class="fas fa-sign-out-alt me-1"></i>Logout</a>
    </div>
    <ul class="nav flex-column">
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'dashboard' %}active{% endif %}" href="{{ url_for('dashboard') }}"><i class="fas fa-tachometer-alt me-2"></i>Dashboard</a></li>
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'map_view' %}active{% endif %}" href="{{ url_for('map_view') }}"><i class="fas fa-map-marked-alt me-2"></i>Map View</a></li>
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'bin_list' %}active{% endif %}" href="{{ url_for('bin_list') }}"><i class="fas fa-trash me-2"></i>Bins</a></li>
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'actions_view' %}active{% endif %}" href="{{ url_for('actions_view') }}"><i class="fas fa-tasks me-2"></i>Actions</a></li>
        {% if current_user.role == 'admin' %}
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'admin_panel' or request.endpoint == 'admin_users' %}active{% endif %}" href="{{ url_for('admin_panel') }}"><i class="fas fa-user-shield me-2"></i>Admin Panel</a></li>
        {% endif %}
    </ul>
</div></nav>
<main class="col-md-9 ms-sm-auto col-lg-10 px-md-4 main-content">
{% block content %}{% endblock %}
</main>
</div></div>
<script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
{% block extra_js %}{% endblock %}
</body>
</html>
'''

with open('templates/base.html', 'w') as f:
    f.write(base_html)


# Create login.html

login_html = '''
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Wakiso Smart Garbage - Login</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        body {
            background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%);
            min-height: 100vh;
            display: flex;
            align-items: center;
            justify-content: center;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        .login-container {
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            padding: 50px 40px;
            max-width: 420px;
            width: 100%;
            animation: fadeInUp 0.6s ease-out;
        }
        @keyframes fadeInUp {
            from { opacity: 0; transform: translateY(30px); }
            to { opacity: 1; transform: translateY(0); }
        }
        .login-header {
            text-align: center;
            margin-bottom: 35px;
        }
        .login-header .icon {
            font-size: 60px;
            color: #2c3e50;
            background: #ecf0f1;
            width: 100px;
            height: 100px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            margin: 0 auto 15px;
        }
        .login-header h2 {
            color: #2c3e50;
            font-weight: 700;
            margin-bottom: 5px;
        }
        .login-header p {
            color: #7f8c8d;
            font-size: 14px;
        }
        .form-control {
            border-radius: 12px;
            padding: 12px 18px;
            border: 2px solid #ecf0f1;
            transition: all 0.3s;
        }
        .form-control:focus {
            border-color: #3498db;
            box-shadow: 0 0 0 3px rgba(52,152,219,0.15);
        }
        .input-group-text {
            background: #ecf0f1;
            border: 2px solid #ecf0f1;
            border-radius: 12px 0 0 12px;
        }
        .btn-login {
            background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%);
            border: none;
            border-radius: 12px;
            padding: 14px;
            font-weight: 600;
            font-size: 16px;
            width: 100%;
            transition: all 0.3s;
            color: white;
        }
        .btn-login:hover {
            transform: translateY(-2px);
            box-shadow: 0 8px 25px rgba(52,152,219,0.4);
        }
        .alert-danger, .alert-success, .alert-info, .alert-warning {
            border-radius: 12px;
        }
        .credentials-box {
            background: #f8f9fa;
            border-radius: 12px;
            padding: 15px;
            margin-top: 20px;
            border: 1px dashed #dee2e6;
        }
        .credentials-box small {
            color: #6c757d;
        }
        .credentials-box .badge {
            font-size: 12px;
        }
        .login-footer {
            text-align: center;
            margin-top: 20px;
            font-size: 13px;
            color: #7f8c8d;
        }
        .demo-credentials {
            font-size: 12px;
            color: #6c757d;
        }
        .demo-credentials .badge {
            margin-right: 3px;
        }
    </style>
</head>
<body>
    <div class="login-container">
        <div class="login-header">
            <div class="icon">
                <i class="fas fa-trash-alt"></i>
            </div>
            <h2>Smart Garbage</h2>
            <p>Wakiso District Operations Control Room</p>
        </div>

        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }} alert-dismissible fade show" role="alert">
                        <i class="fas fa-{% if category == 'success' %}check-circle{% elif category == 'danger' %}exclamation-circle{% else %}info-circle{% endif %} me-2"></i>
                        {{ message }}
                        <button type="button" class="btn-close" data-bs-dismiss="alert"></button>
                    </div>
                {% endfor %}
            {% endif %}
        {% endwith %}

        <form method="post">
            <div class="mb-3">
                <label for="username" class="form-label fw-semibold">Username</label>
                <div class="input-group">
                    <span class="input-group-text"><i class="fas fa-user"></i></span>
                    <input type="text" class="form-control" id="username" name="username"
                           placeholder="Enter your username" required autofocus>
                </div>
            </div>
            <div class="mb-4">
                <label for="password" class="form-label fw-semibold">Password</label>
                <div class="input-group">
                    <span class="input-group-text"><i class="fas fa-lock"></i></span>
                    <input type="password" class="form-control" id="password" name="password"
                           placeholder="Enter your password" required>
                </div>
            </div>
            <button type="submit" class="btn-login">
                <i class="fas fa-sign-in-alt me-2"></i>Sign In
            </button>
        </form>



        <div class="login-footer">
            <i class="fas fa-shield-alt me-1"></i>Secured System &bull; Wakiso District
        </div>
    </div>
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
'''

with open('templates/login.html', 'w') as f:
    f.write(login_html)





# Create map.html

map_html = '''
{% extends 'base.html' %}
{% block title %}Live Map{% endblock %}
{% block extra_css %}
<style>
#map {
    height: 600px;
    width: 100%;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.1);
}
.leaflet-popup-content {
    min-width: 200px;
    max-width: 300px;
}
.legend-item {
    display: inline-block;
    width: 16px;
    height: 16px;
    border-radius: 50%;
    margin-right: 5px;
}
</style>
{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-map-marked-alt me-2"></i>Control Room Map</h1>
<div>
<span class="badge bg-success me-2"><span class="legend-item" style="background:#2ecc71;"></span>Green (0-60%)</span>
<span class="badge bg-warning me-2"><span class="legend-item" style="background:#f39c12;"></span>Amber (61-85%)</span>
<span class="badge bg-danger me-2"><span class="legend-item" style="background:#e74c3c;"></span>Red (86-100%)</span>
<span class="badge bg-info me-2"><span class="legend-item" style="background:#3498db;"></span>Truck</span>
</div>
</div>

<div id="map"></div>

<div class="row mt-3">
    <div class="col-12">
        <div class="card">
            <div class="card-header bg-info text-white">
                <h6><i class="fas fa-info-circle me-2"></i>Map Legend</h6>
            </div>
            <div class="card-body">
                <div class="row">
                    <div class="col-md-3"><span class="badge bg-success">●</span> Green: 0-60% (Operational)</div>
                    <div class="col-md-3"><span class="badge bg-warning">●</span> Amber: 61-85% (Monitor)</div>
                    <div class="col-md-3"><span class="badge bg-danger">●</span> Red: 86-100% (Critical)</div>
                    <div class="col-md-3"><span class="badge bg-info">●</span> Blue: Truck</div>
                </div>
                <div class="row mt-2">
                    <div class="col-md-12"><small class="text-muted">Total Bins: {{ total_bins }} | Total Trucks: {{ total_trucks }}</small></div>
                </div>
            </div>
        </div>
    </div>
</div>
</div>

<script>
// Wait for all resources to load
document.addEventListener('DOMContentLoaded', function() {
    // Check if Leaflet is loaded
    if (typeof L === 'undefined') {
        console.error('Leaflet not loaded!');
        document.getElementById('map').innerHTML = `
            <div style="height:600px;display:flex;align-items:center;justify-content:center;background:#f8f9fa;border-radius:10px;">
                <div class="text-center">
                    <i class="fas fa-spinner fa-spin text-primary" style="font-size:3rem;"></i>
                    <h5 class="mt-3">Loading Map...</h5>
                    <p class="text-muted">Please wait while the map loads.</p>
                </div>
            </div>
        `;
        return;
    }

    // Parse data
    let binData = [];
    let truckData = [];
    try {
        binData = {{ bins_json|safe }};
        truckData = {{ trucks_json|safe }};
    } catch(e) {
        console.error('Error parsing data:', e);
    }

    console.log('Total Bins:', binData.length);
    console.log('Total Trucks:', truckData.length);

    // Initialize map
    const map = L.map('map').setView([0.3476, 32.5825], 12);

    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
        attribution: '© OpenStreetMap contributors',
        maxZoom: 19,
    }).addTo(map);

    const markers = [];

    // Add bin markers
    binData.forEach(bin => {
        if (bin.lat && bin.lng) {
            const color = bin.status_color === 'green' ? '#2ecc71' :
                         bin.status_color === 'amber' ? '#f39c12' : '#e74c3c';

            const icon = L.divIcon({
                className: 'bin-marker',
                html: `<div style="width:28px;height:28px;border-radius:50%;background:${color};border:3px solid white;box-shadow:0 2px 8px rgba(0,0,0,0.4);display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;color:white;">${Math.round(bin.fill)}%</div>`,
                iconSize: [28, 28],
                iconAnchor: [14, 14]
            });

            const marker = L.marker([bin.lat, bin.lng], {icon: icon});

            const popupContent = `
                <div>
                    <h6><i class="fas fa-trash me-1"></i>${bin.name}</h6>
                    <hr style="margin:5px 0;">
                    <p><strong>ID:</strong> ${bin.id}</p>
                    <p><strong>Fill Level:</strong> <span class="fw-bold text-${bin.status_color}">${bin.fill.toFixed(1)}%</span></p>
                    <p><strong>Status:</strong> <span class="badge bg-${bin.status === 'ONLINE' ? 'success' : 'secondary'}">${bin.status}</span></p>
                    <p><strong>Zone:</strong> ${bin.zone}</p>
                    <p><strong>Address:</strong> ${bin.address}</p>
                    <p><strong>Battery:</strong> ${bin.battery.toFixed(1)}%</p>
                    ${bin.last_empty ? `<p><strong>Last Empty:</strong> ${bin.last_empty}</p>` : ''}
                    <a href="/bin/${bin.id}/" class="btn btn-sm btn-primary mt-2">View Details</a>
                </div>
            `;
            marker.bindPopup(popupContent);
            marker.addTo(map);
            markers.push(marker);
        }
    });

    // Add truck markers
    truckData.forEach(truck => {
        if (truck.lat && truck.lng) {
            const color = truck.status === 'IDLE' ? '#3498db' :
                         truck.status === 'EN_ROUTE' ? '#2ecc71' :
                         truck.status === 'COLLECTING' ? '#f39c12' : '#95a5a6';

            const icon = L.divIcon({
                className: 'truck-marker',
                html: `<div style="width:32px;height:32px;border-radius:50%;background:${color};border:3px solid white;display:flex;align-items:center;justify-content:center;color:white;box-shadow:0 2px 8px rgba(0,0,0,0.4);font-size:14px;"><i class="fas fa-truck"></i></div>`,
                iconSize: [32, 32],
                iconAnchor: [16, 16]
            });

            const marker = L.marker([truck.lat, truck.lng], {icon: icon});

            const popupContent = `
                <div>
                    <h6><i class="fas fa-truck me-1"></i>${truck.name}</h6>
                    <hr style="margin:5px 0;">
                    <p><strong>ID:</strong> ${truck.id}</p>
                    <p><strong>Status:</strong> <span class="badge bg-${truck.status === 'IDLE' ? 'info' : truck.status === 'EN_ROUTE' ? 'success' : truck.status === 'COLLECTING' ? 'warning' : 'secondary'}">${truck.status}</span></p>
                    <p><strong>Driver:</strong> ${truck.driver}</p>
                    <p><strong>Speed:</strong> ${truck.speed} km/h</p>
                </div>
            `;
            marker.bindPopup(popupContent);
            marker.addTo(map);
            markers.push(marker);
        }
    });

    console.log(`✅ Added ${markers.length} markers to the map`);

    // Fit bounds to show all markers
    if (markers.length > 0) {
        const group = L.featureGroup(markers);
        map.fitBounds(group.getBounds(), {padding: [50, 50]});
    } else {
        // If no markers, show a message
        document.getElementById('map').innerHTML = `
            <div style="height:600px;display:flex;align-items:center;justify-content:center;background:#f8f9fa;border-radius:10px;">
                <div class="text-center">
                    <i class="fas fa-exclamation-triangle text-warning" style="font-size:3rem;"></i>
                    <h5 class="mt-3">No Data Found</h5>
                    <p class="text-muted">No bins or trucks have been loaded yet.</p>
                    <button class="btn btn-primary btn-sm" onclick="location.reload()">Refresh</button>
                </div>
            </div>
        `;
    }
});

// Handle any errors
window.onerror = function(msg, url, line, col, error) {
    console.error('Map error:', msg);
    return false;
};
</script>
{% endblock %}
'''

with open('templates/map.html', 'w') as f:
    f.write(map_html)


# Create dashboard.html

dashboard_html = '''
{% extends 'base.html' %}
{% block title %}Dashboard{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-tachometer-alt me-2"></i>State of the City</h1>
<div>
<button class="btn btn-sm btn-success me-2" onclick="simulateData()"><i class="fas fa-sync me-1"></i>Simulate IoT Data</button>
<button class="btn btn-sm btn-outline-secondary" onclick="location.reload()"><i class="fas fa-redo me-1"></i>Refresh</button>
</div>
</div>

<div class="row mb-3">
<div class="col-xl-2 col-md-4 col-6"><div class="stats-card"><div class="row"><div class="col-8"><small class="text-muted">Total Bins</small><h3>{{ total_bins }}</h3></div><div class="col-4"><i class="fas fa-trash-alt stats-icon text-primary"></i></div></div></div></div>
<div class="col-xl-2 col-md-4 col-6"><div class="stats-card"><div class="row"><div class="col-8"><small class="text-muted">Critical (Red)</small><h3 class="text-danger">{{ critical_bins }}</h3></div><div class="col-4"><i class="fas fa-exclamation-triangle stats-icon text-danger"></i></div></div></div></div>
<div class="col-xl-2 col-md-4 col-6"><div class="stats-card"><div class="row"><div class="col-8"><small class="text-muted">Trucks On Road</small><h3>{{ trucks_on_road }}/{{ total_trucks }}</h3></div><div class="col-4"><i class="fas fa-truck stats-icon text-success"></i></div></div></div></div>
<div class="col-xl-2 col-md-4 col-6"><div class="stats-card"><div class="row"><div class="col-8"><small class="text-muted">Avg Fill Level</small><h3>{{ avg_fill }}%</h3></div><div class="col-4"><i class="fas fa-chart-line stats-icon text-info"></i></div></div></div></div>
<div class="col-xl-2 col-md-4 col-6"><div class="stats-card"><div class="row"><div class="col-8"><small class="text-muted">Active Alerts</small><h3 class="text-warning">{{ active_alerts }}</h3></div><div class="col-4"><i class="fas fa-bell stats-icon text-warning"></i></div></div></div></div>
<div class="col-xl-2 col-md-4 col-6"><div class="stats-card"><div class="row"><div class="col-8"><small class="text-muted">System Uptime</small><h3>{{ uptime }}%</h3></div><div class="col-4"><i class="fas fa-heart stats-icon text-danger"></i></div></div></div></div>
</div>

<div class="row">
<div class="col-lg-8">
<div class="row">
<div class="col-md-6 mb-3"><div class="card"><div class="card-header"><h6><i class="fas fa-chart-pie me-2"></i>Bin Status</h6></div><div class="card-body"><canvas id="statusChart" height="180"></canvas></div></div></div>
<div class="col-md-6 mb-3"><div class="card"><div class="card-header"><h6><i class="fas fa-chart-bar me-2"></i>Alerts by Type</h6></div><div class="card-body"><canvas id="alertChart" height="180"></canvas></div></div></div>
<div class="col-md-12 mb-3"><div class="card"><div class="card-header"><h6><i class="fas fa-chart-line me-2"></i>24hr Fill Level Trends</h6></div><div class="card-body"><canvas id="fillChart" height="120"></canvas></div></div></div>
<div class="col-md-12 mb-3"><div class="card"><div class="card-header"><h6><i class="fas fa-truck me-2"></i>Truck Status Distribution</h6></div><div class="card-body"><canvas id="truckChart" height="100"></canvas></div></div></div>
</div>
</div>
<div class="col-lg-4">
<div class="card">
<div class="card-header bg-danger text-white">
<h6><i class="fas fa-bell me-2"></i>Real-Time Alert Feed</h6>
</div>
<div class="card-body" id="alertFeed" style="max-height:400px;overflow-y:auto;">
<div class="text-center text-muted">Loading alerts...</div>
</div>
</div>
</div>
</div>
</div>

<script>
async function loadDashboardData() {
    try {
        const r = await fetch('/api/dashboard-data');
        const d = await r.json();

        new Chart(document.getElementById('statusChart'), {
            type: 'doughnut',
            data: {
                labels: ['Online', 'Offline', 'Error'],
                datasets: [{
                    data: [d.status_counts.ONLINE||0, d.status_counts.OFFLINE||0, d.status_counts.ERROR||0],
                    backgroundColor: ['#2ecc71', '#95a5a6', '#e74c3c']
                }]
            },
            options: {responsive: true, plugins: {legend: {position: 'bottom'}}}
        });

        new Chart(document.getElementById('alertChart'), {
            type: 'bar',
            data: {
                labels: ['Full', 'Maintenance', 'Anomaly', 'Low Battery', 'Overflow', 'Truck Delay'],
                datasets: [{
                    label: 'Active Alerts',
                    data: [
                        d.alert_types.FULL||0, d.alert_types.MAINTENANCE||0,
                        d.alert_types.ANOMALY||0, d.alert_types.BATTERY_LOW||0,
                        d.alert_types.OVERFLOW||0, d.alert_types.TRUCK_DELAY||0
                    ],
                    backgroundColor: ['#e74c3c','#f39c12','#9b59b6','#3498db','#c0392b','#e67e22']
                }]
            },
            options: {responsive: true, plugins: {legend: {display: false}}}
        });

        const fillData = d.recent_readings||[];
        if (fillData.length > 0) {
            new Chart(document.getElementById('fillChart'), {
                type: 'line',
                data: {
                    labels: fillData.map(x => x.recorded_at),
                    datasets: [{
                        label: 'Fill Percentage',
                        data: fillData.map(x => x.fill_percentage),
                        borderColor: '#3498db',
                        backgroundColor: 'rgba(52,152,219,0.1)',
                        fill: true,
                        tension: 0.4
                    }]
                },
                options: {responsive: true, scales: {y: {beginAtZero: true, max: 100}}}
            });
        }

        new Chart(document.getElementById('truckChart'), {
            type: 'bar',
            data: {
                labels: ['Idle', 'En Route', 'Collecting', 'Off-Duty'],
                datasets: [{
                    label: 'Trucks',
                    data: [
                        d.truck_status.IDLE||0, d.truck_status.EN_ROUTE||0,
                        d.truck_status.COLLECTING||0, d.truck_status.OFF_DUTY||0
                    ],
                    backgroundColor: ['#3498db', '#2ecc71', '#f39c12', '#95a5a6']
                }]
            },
            options: {responsive: true, plugins: {legend: {display: false}}}
        });

        loadAlertFeed();
    } catch(e) { console.error('Error loading dashboard data:', e); }
}

async function loadAlertFeed() {
    try {
        const r = await fetch('/api/alerts-feed');
        const data = await r.json();
        const feed = document.getElementById('alertFeed');

        if (data.alerts.length === 0) {
            feed.innerHTML = '<div class="text-center text-muted"><i class="fas fa-check-circle text-success me-2"></i>All clear! No active alerts.</div>';
            return;
        }

        let html = '';
        data.alerts.forEach(alert => {
            const priorityClass = alert.priority === 'CRITICAL' ? 'danger' :
                                 alert.priority === 'HIGH' ? 'warning' :
                                 alert.priority === 'MEDIUM' ? 'info' : 'secondary';
            html += `<div class="alert alert-${priorityClass} alert-dismissible fade show" role="alert">
                <small class="float-end">${alert.created_at}</small>
                <div><strong>${alert.type}</strong> - ${alert.source}</div>
                <div>${alert.message}</div>
                ${!alert.is_acknowledged ? `<button class="btn btn-sm btn-outline-primary mt-1" onclick="acknowledgeAlert(${alert.id})"><i class="fas fa-check me-1"></i>Acknowledge</button>` : '<span class="badge bg-success"><i class="fas fa-check me-1"></i>Acknowledged</span>'}
            </div>`;
        });
        feed.innerHTML = html;
    } catch(e) { console.error('Error loading alerts:', e); }
}

async function acknowledgeAlert(id) {
    try {
        await fetch(`/api/acknowledge/${id}`, {method: 'POST'});
        loadAlertFeed();
    } catch(e) { console.error(e); }
}

async function simulateData() {
    try {
        const r = await fetch('/api/simulate', {method: 'POST'});
        const result = await r.json();
        if (result.status === 'Data simulated successfully') {
            location.reload();
        }
    } catch(e) { console.error(e); }
}

document.addEventListener('DOMContentLoaded', function() {
    loadDashboardData();
    setInterval(loadAlertFeed, 30000);
});
</script>
{% endblock %}
'''

with open('templates/dashboard.html', 'w') as f:
    f.write(dashboard_html)


# Create bin_list.html

bin_list_html = '''
{% extends 'base.html' %}
{% block title %}Bins{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-trash me-2"></i>Smart Bins</h1>
<div class="input-group" style="width:300px;"><input type="text" class="form-control" placeholder="Search..." id="searchInput"><button class="btn btn-outline-secondary"><i class="fas fa-search"></i></button></div>
</div>

<div class="row mb-3">
<div class="col-md-3"><select class="form-select" id="zoneFilter"><option value="">All Zones</option>{% for zone in zones %}<option value="{{ zone.id }}">{{ zone.name }}</option>{% endfor %}</select></div>
<div class="col-md-3"><select class="form-select" id="statusFilter"><option value="">All Status</option><option value="ONLINE">Online</option><option value="OFFLINE">Offline</option><option value="ERROR">Error</option></select></div>
<div class="col-md-3"><select class="form-select" id="fillFilter"><option value="">All Fill Levels</option><option value="green">Green (0-60%)</option><option value="amber">Amber (61-85%)</option><option value="red">Red (86-100%)</option></select></div>
<div class="col-md-3"><select class="form-select" id="typeFilter"><option value="">All Types</option><option value="STANDARD">Standard</option><option value="RECYCLING">Recycling</option><option value="ORGANIC">Organic</option><option value="MEDICAL">Medical</option><option value="ELECTRONIC">Electronic</option></select></div>
</div>

<div class="row" id="binContainer">
{% for bin in bins %}
<div class="col-md-4 mb-3 bin-card"
     data-zone="{{ bin.zone_id }}"
     data-status="{{ bin.status }}"
     data-fill="{{ bin.get_status_color() }}"
     data-type="{{ bin.bin_type }}">
<div class="card">
<div class="card-body">
<div class="d-flex justify-content-between">
<h5 class="card-title">{{ bin.name }}</h5>
<span class="badge bg-{{ bin.get_fill_level_class() }}">{{ bin.fill_percentage|round(0)|int }}%</span>
</div>
<div class="mb-2">
<small class="text-muted">Zone: {{ bin.zone.name if bin.zone else 'Unknown' }}</small><br>
<small class="text-muted">Status: <span class="badge bg-{% if bin.status == 'ONLINE' %}success{% elif bin.status == 'OFFLINE' %}secondary{% else %}danger{% endif %}">{{ bin.status }}</span></small>
</div>
<div class="progress mb-2" style="height:25px;">
<div class="progress-bar bg-{{ bin.get_fill_level_class() }}" style="width:{{ bin.fill_percentage }}%">{{ bin.fill_percentage|round(0)|int }}%</div>
</div>
<div class="d-flex justify-content-between">
<small>{{ bin.current_fill_level|round(0)|int }}/{{ bin.capacity|round(0)|int }} L</small>
<small><i class="fas fa-battery-{% if bin.battery_level > 75 %}full{% elif bin.battery_level > 50 %}three-quarters{% elif bin.battery_level > 25 %}quarter{% else %}empty{% endif %}"></i> {{ bin.battery_level|round(0)|int }}%</small>
<a href="{{ url_for('bin_detail', bin_id=bin.bin_id) }}" class="btn btn-sm btn-primary">Details</a>
</div>
</div>
</div>
</div>
{% endfor %}
</div>
</div>

<script>
document.addEventListener('DOMContentLoaded',function(){
const s=document.getElementById('searchInput'),z=document.getElementById('zoneFilter'),st=document.getElementById('statusFilter'),f=document.getElementById('fillFilter'),t=document.getElementById('typeFilter'),cards=document.querySelectorAll('.bin-card');
function filter(){const term=s.value.toLowerCase(),zone=z.value,status=st.value,fill=f.value,type=t.value;
cards.forEach(c=>{const name=c.querySelector('.card-title').textContent.toLowerCase(),show=!(term&&!name.includes(term))&&!(zone&&c.dataset.zone!==zone)&&!(status&&c.dataset.status!==status)&&!(fill&&c.dataset.fill!==fill)&&!(type&&c.dataset.type!==type);c.style.display=show?'block':'none';});}
s.addEventListener('input',filter);z.addEventListener('change',filter);st.addEventListener('change',filter);f.addEventListener('change',filter);t.addEventListener('change',filter);
});
</script>
{% endblock %}
'''

with open('templates/bin_list.html', 'w') as f:
    f.write(bin_list_html)



# Create bin_detail.html

bin_detail_html = '''
{% extends 'base.html' %}
{% block title %}{{ bin.name }}{% endblock %}
{% block content %}
<div>
<nav aria-label="breadcrumb"><ol class="breadcrumb"><li class="breadcrumb-item"><a href="{{ url_for('dashboard') }}">Dashboard</a></li><li class="breadcrumb-item"><a href="{{ url_for('bin_list') }}">Bins</a></li><li class="breadcrumb-item active">{{ bin.name }}</li></ol></nav>

<div class="row">
<div class="col-md-8">
<div class="card">
<div class="card-header"><h5><i class="fas fa-trash me-2"></i>{{ bin.name }}</h5></div>
<div class="card-body">
<div class="row mb-3">
<div class="col-md-6">
<p><strong>Bin ID:</strong> {{ bin.bin_id }}</p>
<p><strong>Type:</strong> {{ bin.bin_type }}</p>
<p><strong>Status:</strong> <span class="badge bg-{% if bin.status == 'ONLINE' %}success{% elif bin.status == 'OFFLINE' %}secondary{% else %}danger{% endif %}">{{ bin.status }}</span></p>
<p><strong>Battery:</strong> {{ bin.battery_level|round(0)|int }}%</p>
</div>
<div class="col-md-6">
<p><strong>Zone:</strong> {{ bin.zone.name if bin.zone else 'Unknown' }}</p>
<p><strong>Address:</strong> {{ bin.address }}</p>
<p><strong>Last Empty:</strong> {{ bin.last_empty.strftime('%Y-%m-%d %H:%M') if bin.last_empty else 'Never' }}</p>
</div>
</div>

<div class="mb-3">
<h6>Fill Level</h6>
<div class="progress" style="height:35px;">
<div class="progress-bar bg-{{ bin.get_fill_level_class() }}" style="width:{{ bin.fill_percentage }}%">
{{ bin.fill_percentage|round(0)|int }}% ({{ bin.current_fill_level|round(0)|int }}/{{ bin.capacity|round(0)|int }} L)
</div>
</div>
</div>

<div class="row">
<div class="col-md-4"><p><strong>Temperature:</strong> {{ bin.temperature if bin.temperature else 'N/A' }}°C</p></div>
<div class="col-md-4"><p><strong>Humidity:</strong> {{ bin.humidity if bin.humidity else 'N/A' }}%</p></div>
<div class="col-md-4"><p><strong>Last Maintenance:</strong> {{ bin.last_maintenance.strftime('%Y-%m-%d') if bin.last_maintenance else 'Never' }}</p></div>
</div>
</div>
</div>
</div>

<div class="col-md-4">
<div class="card">
<div class="card-header bg-danger text-white"><h6><i class="fas fa-exclamation-triangle me-2"></i>Active Alerts</h6></div>
<div class="card-body">
{% if alerts %}
{% for alert in alerts %}
<div class="alert alert-{% if alert.priority == 'CRITICAL' %}danger{% elif alert.priority == 'HIGH' %}warning{% else %}info{% endif %}">
<strong>{{ alert.alert_type }}</strong>
<p>{{ alert.message }}</p>
<small class="text-muted">{{ alert.created_at|timesince }} ago</small>
</div>
{% endfor %}
{% else %}
<p class="text-center text-muted">No active alerts</p>
{% endif %}
</div>
</div>
</div>
</div>

<div class="row mt-3">
<div class="col-12">
<div class="card">
<div class="card-header"><h6><i class="fas fa-chart-line me-2"></i>Fill Level History</h6></div>
<div class="card-body"><canvas id="historyChart" height="150"></canvas></div>
</div>
</div>
</div>
</div>

<script>
document.addEventListener('DOMContentLoaded',function(){
const readings=[{% for reading in readings|reverse %}{time:"{{ reading.recorded_at.strftime('%H:%M') }}",fill:{{ reading.fill_percentage }}},{% endfor %}];
if (readings.length > 0) {
    new Chart(document.getElementById('historyChart'),{type:'line',data:{labels:readings.map(r=>r.time),datasets:[{label:'Fill Percentage',data:readings.map(r=>r.fill),borderColor:'#3498db',backgroundColor:'rgba(52,152,219,0.1)',fill:true,tension:0.4}]},options:{responsive:true,scales:{y:{beginAtZero:true,max:100}}}});
} else {
    document.getElementById('historyChart').parentElement.innerHTML = '<p class="text-center text-muted">No historical data available</p>';
}
});
</script>
{% endblock %}
'''

with open('templates/bin_detail.html', 'w') as f:
    f.write(bin_detail_html)


# CREATE ACTIONS.HTML

actions_html = '''
{% extends 'base.html' %}
{% block title %}Actions{% endblock %}
{% block content %}
<div>
<div class="d-flex justify-content-between flex-wrap flex-md-nowrap align-items-center pb-2 mb-3 border-bottom">
<h1 class="h2"><i class="fas fa-tasks me-2"></i>Prioritized Action List</h1>
<button class="btn btn-sm btn-primary" onclick="loadActions()"><i class="fas fa-redo me-1"></i>Refresh</button>
</div>

<div class="row">
<div class="col-12">
<div class="card">
<div class="card-header bg-primary text-white">
<h6><i class="fas fa-clock me-2"></i>Urgency Queue - Sort by Priority</h6>
</div>
<div class="card-body" id="actionsTable">
<div class="text-center text-muted">Loading actions...</div>
</div>
</div>
</div>
</div>

<div class="row mt-3">
<div class="col-12">
<div class="card">
<div class="card-header bg-success text-white">
<h6><i class="fas fa-route me-2"></i>Route Optimization Suggestion</h6>
</div>
<div class="card-body" id="routeSuggestion">
<div class="text-center text-muted">Calculating optimal routes...</div>
</div>
</div>
</div>
</div>
</div>

<script>
async function loadActions() {
    try {
        const r = await fetch('/api/actions-data');
        const data = await r.json();
        const container = document.getElementById('actionsTable');

        if (data.actions.length === 0) {
            container.innerHTML = '<div class="alert alert-success"><i class="fas fa-check-circle me-2"></i>All bins are in good condition! No urgent actions required.</div>';
            return;
        }

        let html = `<table class="table table-hover">
            <thead><tr>
                <th>Priority</th><th>Bin ID</th><th>Location</th><th>Fill %</th><th>Assigned Truck</th><th>Action</th>
            </tr></thead><tbody>`;

        data.actions.forEach(action => {
            const priorityClass = action.priority === 'CRITICAL' ? 'danger' :
                                 action.priority === 'HIGH' ? 'warning' : 'info';
            const icon = action.priority === 'CRITICAL' ? 'fa-bell' :
                        action.priority === 'HIGH' ? 'fa-exclamation-triangle' : 'fa-info-circle';

            html += `<tr>
                <td><span class="badge bg-${priorityClass}"><i class="fas ${icon} me-1"></i>${action.priority}</span></td>
                <td><strong>${action.bin_id}</strong></td>
                <td>${action.name}<br><small class="text-muted">${action.zone}</small></td>
                <td><span class="fw-bold text-${action.fill > 90 ? 'danger' : 'warning'}">${action.fill}%</span></td>
                <td>${action.assigned_truck}</td>
                <td>
                    <button class="btn btn-sm btn-danger dispatch-btn" data-bin="${action.bin_id}">
                        <i class="fas fa-truck me-1"></i>Dispatch Now
                    </button>
                </td>
            </tr>`;
        });

        html += '</tbody></table>';
        container.innerHTML = html;

        document.querySelectorAll('.dispatch-btn').forEach(btn => {
            btn.addEventListener('click', function() {
                alert(`Truck dispatched to bin ${this.dataset.bin}!`);
                this.innerHTML = '<i class="fas fa-check me-1"></i>Dispatched';
                this.className = 'btn btn-sm btn-success dispatch-btn';
            });
        });

        if (data.actions.length > 0) {
            const top3 = data.actions.slice(0, 3);
            document.getElementById('routeSuggestion').innerHTML = `
                <div class="alert alert-success">
                    <i class="fas fa-lightbulb me-2"></i>
                    <strong>Optimized Route Recommendation:</strong><br>
                    Truck should collect bins in this sequence for maximum efficiency:<br>
                    <ol class="mt-2">
                        ${top3.map((a, i) => `<li><strong>${a.bin_id}</strong> - ${a.name} (${a.fill}%) - Priority: ${a.priority}</li>`).join('')}
                    </ol>
                    <small class="text-muted">Estimated time savings: ~15 minutes</small>
                </div>
            `;
        }

    } catch(e) { console.error(e); }
}

document.addEventListener('DOMContentLoaded', loadActions);
</script>
{% endblock %}
'''

with open('templates/actions.html', 'w') as f:
    f.write(actions_html)



# RUN THE APPLICATION


def run_flask():
    os.system('python app.py')

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(10)

# DISPLAY OUTPUT


print("🎉 WAKISO SMART GARBAGE SYSTEM IS READY!")

print("\n📍 To access the app:")
print("   Click the 'port 8000' link below")



from google.colab import output
output.serve_kernel_port_as_window(8000)

print("\n⚠️  Keep this notebook running to access the app!")

🔄 Cleaning up existing processes...
🎉 WAKISO SMART GARBAGE SYSTEM IS READY!

📍 To access the app:
   Click the 'port 8000' link below
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>


⚠️  Keep this notebook running to access the app!
